# 02 - Perfilamiento de esquemas

## Objetivo
Comprobar contra los datos reales (no solo contra la documentación de TLC) qué columnas tiene cada uno de los 4 tipos de taxi, detectar si hay diferencias de esquema entre años (schema drift), y construir un mapeo de columnas canónico que usará el notebook `03_etl_limpieza_spark` para normalizar los 4 tipos.

**Prerrequisitos:**
- Spark funcionando en esta instancia EC2 (kernel Python 3.12 Spark), con acceso a S3 vía `s3a://`.
- Los 117 objetos ya están en `raw_data/` (no depende de que el backfill de metadata haya terminado — son procesos independientes).

**Salida esperada:** un JSON en `s3://xideralaws-curso-proyecto-alan/schema_profile/schema_profile.json` con el esquema real detectado por tipo y el mapeo de columnas propuesto para las capas Silver/Gold.

In [1]:
%pip install boto3

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pyspark.sql import SparkSession
import boto3
import json
from datetime import datetime, timezone

BUCKET = "xideralaws-curso-proyecto-alan"

config = {
    "spark.jars.packages": "org.apache.hadoop:hadoop-aws:3.4.2,software.amazon.awssdk:bundle:2.29.52",
    "spark.hadoop.fs.s3a.aws.credentials.provider": "software.amazon.awssdk.auth.credentials.ProfileCredentialsProvider",
    "spark.hadoop.fs.s3a.endpoint.region": "us-west-1"
}
spark = SparkSession.builder.appName("perfilamientoEsquemas").config(map=config).getOrCreate()

:: loading settings :: url = jar:file:/home/ubuntu/opt/spark-4.1.2-bin-hadoop3/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/ubuntu/.ivy2.5.2/cache
The jars for the packages stored in: /home/ubuntu/.ivy2.5.2/jars
org.apache.hadoop#hadoop-aws added as a dependency
software.amazon.awssdk#bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-6c01383e-aa1c-4255-9fab-6bba10d0e289;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.4.2 in central
	found software.amazon.awssdk#bundle;2.29.52 in central
	found software.amazon.s3.analyticsaccelerator#analyticsaccelerator-s3;1.2.1 in central
	found org.wildfly.openssl#wildfly-openssl;2.1.4.Final in central
:: resolution report :: resolve 424ms :: artifacts dl 15ms
	:: modules in use:
	org.apache.hadoop#hadoop-aws;3.4.2 from central in [default]
	org.wildfly.openssl#wildfly-openssl;2.1.4.Final from central in [default]
	software.amazon.awssdk#bundl

### Estrategia de perfilamiento
Para cada tipo de taxi, leemos TODOS sus archivos (los 3 años) con `mergeSchema=true` usando un patrón con wildcard. Esto no carga las filas en memoria: Spark solo lee el footer de metadata de cada Parquet para construir la unión de columnas — es información, no datos.

**Por qué así y no archivo por archivo:** si un año introdujo o quitó una columna (por ejemplo, TLC agregó campos nuevos en algunos años), `mergeSchema` lo revela automáticamente como parte de la unión, sin que tengamos que inspeccionar los 117 archivos uno por uno a mano.

In [3]:
taxi_types = ["yellow", "green", "fhv", "fhvhv"]
esquemas = {}

for tipo in taxi_types:
    path = f"s3a://{BUCKET}/raw_data/*/*/{tipo}_tripdata_*.parquet"
    df = spark.read.option("mergeSchema", "true").parquet(path)
    esquemas[tipo] = df.schema

    print(f"--- {tipo} ({len(df.schema.fields)} columnas) ---")
    for campo in df.schema.fields:
        print(f"  {campo.name}: {campo.dataType.simpleString()}")
    print()

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.
26/09/18 04:58:45 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: s3a://xideralaws-curso-proyecto-alan/raw_data/*/*/yellow_tripdata_*.parquet.
java.io.FileNotFoundException: No such file or directory: s3a://xideralaws-curso-proyecto-alan/raw_data/*/*/yellow_tripdata_*.parquet
	at org.apache.hadoop.fs.s3a.S3AFileSystem.s3GetFileStatus(S3AFileSystem.java:4169)
	at org.apache.hadoop.fs.s3a.S3AFileSystem.innerGetFileStatus(S3AFileSystem.java:4027)
	at org.apache.hadoop.fs.s3a.S3AFileSystem.lambda$getFileStatus$21(S3AFileSystem.java:4004)
	at org.apache.hadoop.fs.statistics.impl.IOStatisticsBinding.invokeTrackingDuration(IOStatisticsBinding.java:547)
	at org.apache.hadoop.fs.statistics.impl.IOStatisticsBinding.lambda$trackDura

--- yellow (20 columnas) ---
  VendorID: int
  tpep_pickup_datetime: timestamp_ntz
  tpep_dropoff_datetime: timestamp_ntz
  passenger_count: bigint
  trip_distance: double
  RatecodeID: bigint
  store_and_fwd_flag: string
  PULocationID: int
  DOLocationID: int
  payment_type: bigint
  fare_amount: double
  extra: double
  mta_tax: double
  tip_amount: double
  tolls_amount: double
  improvement_surcharge: double
  total_amount: double
  congestion_surcharge: double
  Airport_fee: double
  cbd_congestion_fee: double



26/09/18 04:58:49 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: s3a://xideralaws-curso-proyecto-alan/raw_data/*/*/green_tripdata_*.parquet.
java.io.FileNotFoundException: No such file or directory: s3a://xideralaws-curso-proyecto-alan/raw_data/*/*/green_tripdata_*.parquet
	at org.apache.hadoop.fs.s3a.S3AFileSystem.s3GetFileStatus(S3AFileSystem.java:4169)
	at org.apache.hadoop.fs.s3a.S3AFileSystem.innerGetFileStatus(S3AFileSystem.java:4027)
	at org.apache.hadoop.fs.s3a.S3AFileSystem.lambda$getFileStatus$21(S3AFileSystem.java:4004)
	at org.apache.hadoop.fs.statistics.impl.IOStatisticsBinding.invokeTrackingDuration(IOStatisticsBinding.java:547)
	at org.apache.hadoop.fs.statistics.impl.IOStatisticsBinding.lambda$trackDurationOfOperation$5(IOStatisticsBinding.java:528)
	at org.apache.hadoop.fs.statistics.impl.IOStatisticsBinding.trackDuration(IOStatisticsBinding.java:449)
	at org.apache.hadoop.fs.s3a.S3AFileSystem.trackDurationAnd

--- green (21 columnas) ---
  VendorID: int
  lpep_pickup_datetime: timestamp_ntz
  lpep_dropoff_datetime: timestamp_ntz
  store_and_fwd_flag: string
  RatecodeID: bigint
  PULocationID: int
  DOLocationID: int
  passenger_count: bigint
  trip_distance: double
  fare_amount: double
  extra: double
  mta_tax: double
  tip_amount: double
  tolls_amount: double
  ehail_fee: double
  improvement_surcharge: double
  total_amount: double
  payment_type: bigint
  trip_type: bigint
  congestion_surcharge: double
  cbd_congestion_fee: double



26/09/18 04:58:51 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: s3a://xideralaws-curso-proyecto-alan/raw_data/*/*/fhv_tripdata_*.parquet.
java.io.FileNotFoundException: No such file or directory: s3a://xideralaws-curso-proyecto-alan/raw_data/*/*/fhv_tripdata_*.parquet
	at org.apache.hadoop.fs.s3a.S3AFileSystem.s3GetFileStatus(S3AFileSystem.java:4169)
	at org.apache.hadoop.fs.s3a.S3AFileSystem.innerGetFileStatus(S3AFileSystem.java:4027)
	at org.apache.hadoop.fs.s3a.S3AFileSystem.lambda$getFileStatus$21(S3AFileSystem.java:4004)
	at org.apache.hadoop.fs.statistics.impl.IOStatisticsBinding.invokeTrackingDuration(IOStatisticsBinding.java:547)
	at org.apache.hadoop.fs.statistics.impl.IOStatisticsBinding.lambda$trackDurationOfOperation$5(IOStatisticsBinding.java:528)
	at org.apache.hadoop.fs.statistics.impl.IOStatisticsBinding.trackDuration(IOStatisticsBinding.java:449)
	at org.apache.hadoop.fs.s3a.S3AFileSystem.trackDurationAndSpan

--- fhv (7 columnas) ---
  dispatching_base_num: string
  pickup_datetime: timestamp_ntz
  dropOff_datetime: timestamp_ntz
  PUlocationID: bigint
  DOlocationID: bigint
  SR_Flag: bigint
  Affiliated_base_number: string



26/09/18 04:58:53 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: s3a://xideralaws-curso-proyecto-alan/raw_data/*/*/fhvhv_tripdata_*.parquet.
java.io.FileNotFoundException: No such file or directory: s3a://xideralaws-curso-proyecto-alan/raw_data/*/*/fhvhv_tripdata_*.parquet
	at org.apache.hadoop.fs.s3a.S3AFileSystem.s3GetFileStatus(S3AFileSystem.java:4169)
	at org.apache.hadoop.fs.s3a.S3AFileSystem.innerGetFileStatus(S3AFileSystem.java:4027)
	at org.apache.hadoop.fs.s3a.S3AFileSystem.lambda$getFileStatus$21(S3AFileSystem.java:4004)
	at org.apache.hadoop.fs.statistics.impl.IOStatisticsBinding.invokeTrackingDuration(IOStatisticsBinding.java:547)
	at org.apache.hadoop.fs.statistics.impl.IOStatisticsBinding.lambda$trackDurationOfOperation$5(IOStatisticsBinding.java:528)
	at org.apache.hadoop.fs.statistics.impl.IOStatisticsBinding.trackDuration(IOStatisticsBinding.java:449)
	at org.apache.hadoop.fs.s3a.S3AFileSystem.trackDurationAnd

--- fhvhv (25 columnas) ---
  hvfhs_license_num: string
  dispatching_base_num: string
  originating_base_num: string
  request_datetime: timestamp_ntz
  on_scene_datetime: timestamp_ntz
  pickup_datetime: timestamp_ntz
  dropoff_datetime: timestamp_ntz
  PULocationID: int
  DOLocationID: int
  trip_miles: double
  trip_time: bigint
  base_passenger_fare: double
  tolls: double
  bcf: double
  sales_tax: double
  congestion_surcharge: double
  airport_fee: double
  tips: double
  driver_pay: double
  shared_request_flag: string
  shared_match_flag: string
  access_a_ride_flag: string
  wav_request_flag: string
  wav_match_flag: string
  cbd_congestion_fee: double



### Comparación entre los 4 tipos
Comparamos los nombres de columna en minúsculas (para que `PULocationID` de yellow/green/fhvhv y `PUlocationID` de fhv se reconozcan como la misma columna pese a la diferencia de mayúsculas) y calculamos la intersección entre los 4 tipos.

In [4]:
nombres_por_tipo = {
    tipo: {campo.name.lower() for campo in esquemas[tipo].fields}
    for tipo in taxi_types
}

comunes = set.intersection(*nombres_por_tipo.values())
print("Columnas presentes en los 4 tipos (comparando en minúsculas):")
print(sorted(comunes))

print()
for tipo in taxi_types:
    exclusivas = nombres_por_tipo[tipo] - comunes
    print(f"Columnas de {tipo} que NO están en los 4 tipos: {sorted(exclusivas)}")

Columnas presentes en los 4 tipos (comparando en minúsculas):
['dolocationid', 'pulocationid']

Columnas de yellow que NO están en los 4 tipos: ['airport_fee', 'cbd_congestion_fee', 'congestion_surcharge', 'extra', 'fare_amount', 'improvement_surcharge', 'mta_tax', 'passenger_count', 'payment_type', 'ratecodeid', 'store_and_fwd_flag', 'tip_amount', 'tolls_amount', 'total_amount', 'tpep_dropoff_datetime', 'tpep_pickup_datetime', 'trip_distance', 'vendorid']
Columnas de green que NO están en los 4 tipos: ['cbd_congestion_fee', 'congestion_surcharge', 'ehail_fee', 'extra', 'fare_amount', 'improvement_surcharge', 'lpep_dropoff_datetime', 'lpep_pickup_datetime', 'mta_tax', 'passenger_count', 'payment_type', 'ratecodeid', 'store_and_fwd_flag', 'tip_amount', 'tolls_amount', 'total_amount', 'trip_distance', 'trip_type', 'vendorid']
Columnas de fhv que NO están en los 4 tipos: ['affiliated_base_number', 'dispatching_base_num', 'dropoff_datetime', 'pickup_datetime', 'sr_flag']
Columnas de fhvhv 

### Mapeo canónico propuesto
Con base en lo anterior, definimos dos niveles de esquema (ver decisión de arquitectura: `core_trips` con los 4 tipos, `economic_trips` solo con los que tienen fare/distancia/propina comparables):

- `core_trips`: `pickup_datetime`, `dropoff_datetime`, `PULocationID`, `DOLocationID` — los 4 tipos.
- `economic_trips`: además `distance`, `fare`, `tip` — solo yellow, green, fhvhv (fhv no tiene datos económicos comparables).

El mapeo no se escribe a ciegas: la celda siguiente valida cada nombre contra las columnas realmente detectadas arriba, y avisa si algo no coincide (por ejemplo, si un año futuro cambiara un nombre de columna).

In [5]:
mapeo_canonico = {
    "yellow": {"pickup_datetime": "tpep_pickup_datetime", "dropoff_datetime": "tpep_dropoff_datetime",
               "PULocationID": "PULocationID", "DOLocationID": "DOLocationID"},
    "green":  {"pickup_datetime": "lpep_pickup_datetime", "dropoff_datetime": "lpep_dropoff_datetime",
               "PULocationID": "PULocationID", "DOLocationID": "DOLocationID"},
    "fhv":    {"pickup_datetime": "pickup_datetime", "dropoff_datetime": "dropOff_datetime",
               "PULocationID": "PUlocationID", "DOLocationID": "DOlocationID"},
    "fhvhv":  {"pickup_datetime": "pickup_datetime", "dropoff_datetime": "dropoff_datetime",
               "PULocationID": "PULocationID", "DOLocationID": "DOLocationID"},
}

mapeo_economico = {
    "yellow": {"distance": "trip_distance", "fare": "fare_amount", "tip": "tip_amount"},
    "green":  {"distance": "trip_distance", "fare": "fare_amount", "tip": "tip_amount"},
    "fhvhv":  {"distance": "trip_miles", "fare": "base_passenger_fare", "tip": "tips"},
}

# Validación: cada nombre de columna referenciado debe existir realmente en el esquema detectado
for tipo, mapeo in mapeo_canonico.items():
    columnas_reales = {c.name for c in esquemas[tipo].fields}
    for campo_canonico, nombre_real in mapeo.items():
        if nombre_real not in columnas_reales:
            print(f"ADVERTENCIA: {tipo} no tiene la columna esperada '{nombre_real}' para '{campo_canonico}'")

for tipo, mapeo in mapeo_economico.items():
    columnas_reales = {c.name for c in esquemas[tipo].fields}
    for campo_canonico, nombre_real in mapeo.items():
        if nombre_real not in columnas_reales:
            print(f"ADVERTENCIA: {tipo} no tiene la columna esperada '{nombre_real}' para '{campo_canonico}'")

print("Validación completa (si no ves ADVERTENCIA arriba, el mapeo coincide con los datos reales).")

Validación completa (si no ves ADVERTENCIA arriba, el mapeo coincide con los datos reales).


### Guardar el perfil en S3
Para que `03_etl_limpieza_spark` no tenga que repetir este perfilamiento, guardamos el resultado (esquemas detectados + mapeos) como JSON. Además, será parte de la documentación del pipeline.

In [6]:
perfil = {
    "generado_en": datetime.now(timezone.utc).isoformat(),
    "esquemas_detectados": {
        tipo: [{"nombre": c.name, "tipo": c.dataType.simpleString()} for c in esquemas[tipo].fields]
        for tipo in taxi_types
    },
    "columnas_core": mapeo_canonico,
    "columnas_economicas": mapeo_economico,
}

s3_client = boto3.client("s3", region_name="us-west-1")
s3_client.put_object(
    Bucket=BUCKET,
    Key="schema_profile/schema_profile.json",
    Body=json.dumps(perfil, indent=2, default=str).encode("utf-8"),
    ContentType="application/json"
)

print("Perfil guardado en s3://" + BUCKET + "/schema_profile/schema_profile.json")

Perfil guardado en s3://xideralaws-curso-proyecto-alan/schema_profile/schema_profile.json


### Liberar memoria
La t2.medium tiene poca RAM, por lo tanto, para evitar saturarla, cerramos la SparkSession antes de abrir el siguiente notebook para no dejarla ocupando memoria de fondo.

In [7]:
spark.stop()